# BusinessGPT ORPO Fine-Tuning: Qwen3.5-9B (abliterated) — preference learning, single-stage

Replaces DPO. Trains a LoRA adapter that combines SFT-on-chosen + odds-ratio preference loss in ONE stage. No reference model, no precompute, no log-ratio drift.

**Why ORPO over DPO** (3 DPO failures: v13-dpo NaN, v14-dpo gay-spam regression, v15-dpo abandoned):
- No ref model → no precompute_ref_log_probs (that was the fp16 NaN trigger on Qwen3.5 hybrid linear_attn).
- No log-ratio → no off-policy drift, no length-bias amplification.
- Single loss: `L = L_SFT(chosen) + λ · L_OR(chosen, rejected)` where `L_OR = -log σ(log odds(chosen) - log odds(rejected))`.
- Reference: huggingface.co/blog/orpo, Hong et al 2024.

**Pipeline position:**

1. SFT (`training.ipynb`) → v16 SFT model on HF
2. Multi-candidate eval generates 4 candidates per prompt → `eval/generations_v14_multi.json` (already done for v14)
3. Pairwise labeling → `eval/ratings_v14_multi.json` (already done, 1450 pairs)
4. `build_dpo_from_multi("v14")` → `eval/preference_pairs_v14_multi.jsonl` (already done)
5. Upload to Kaggle dataset `businessgpt-eval`
6. **This notebook** → ORPO model on HF
7. Eval cell at end → `eval/generations_v16-orpo{,_multi}.json` for bench comparison

Set `BASE_VERSION` / `ORPO_VERSION` / `PAIRS_VERSION` in the config cell before running.

**Reload from HF only:** set `LOAD_ORPO_FROM_HF = True`, run **Load ORPO from Hugging Face**, then section 7 — skip preference pairs and training (sections 2–6).

**Requirements**: Kaggle GPU T4×2; preference pairs file in attached `businessgpt-eval` dataset.

## 0. Install Dependencies

In [ ]:
%%capture
%pip install vllm --torch-backend=auto --extra-index-utorl https://wheels.vllm.ai/nightly
%pip install git+https://github.com/huggingface/transformers.git
# %pip install git+https://github.com/huggingface/peft.git
%pip install -U huggingface_hub trl kagglehub datasets accelerate bitsandbytes peft torchao

In [ ]:
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("HF_TOKEN")
login(token=hf_token)

## 1. Configuration

In [ ]:
# Edit these for each ORPO run.
BASE_VERSION = "v16"            # SFT version to start from (the merged base for ORPO LoRA)
ORPO_VERSION = "v16-orpo"       # output name for the ORPO adapter
PAIRS_VERSION = "v14"           # which labeled set to use (v14_multi has 1450 pairs)
SUPER_WEIGHT = 1                # multi-candidate data has no super tier; ignored

# Data source: True = in-distribution v<PAIRS_VERSION>_multi pairs (recommended).
# False = legacy cross-version preference_pairs.jsonl (off-policy; previously collapsed under DPO).
USE_MULTI_PAIRS = True

# Auto-derive size suffix from BASE_VERSION naming convention.
# v11..v14 → 2b, v15+ → 9b. Override _MODEL_SIZE manually if needed.
_MODEL_SIZE = "9b" if int(__import__("re").match(r"v(\d+)", BASE_VERSION).group(1)) >= 15 else "2b"
_BASE_NAME  = f"Huihui-Qwen3.5-{_MODEL_SIZE.upper()}-abliterated"

BASE_MODEL_ID = f"huihui-ai/{_BASE_NAME}"
SFT_REPO  = f"vXofi/businessgpt-{BASE_VERSION}-qwen3.5-{_MODEL_SIZE}"
ORPO_REPO = f"vXofi/businessgpt-{ORPO_VERSION}-qwen3.5-{_MODEL_SIZE}"
SAVE_DIR  = f"businessgpt_{ORPO_VERSION}_{_MODEL_SIZE}_model"

# Inference-only: set True and run "Load ORPO from Hugging Face" after this cell, then jump to section 7 (eval).
# Training cells will raise if you forget and run them with this flag on.
LOAD_ORPO_FROM_HF = False

print(f"BASE: {SFT_REPO}  ({BASE_MODEL_ID}, size={_MODEL_SIZE})")
print(f"OUT:  {ORPO_REPO}")
print(f"DATA: preference_pairs_{PAIRS_VERSION}_multi.jsonl ({'in-distribution' if USE_MULTI_PAIRS else 'cross-version (off-policy)'})")

## Load ORPO from Hugging Face (inference-only)

If you **already pushed** the ORPO adapter to Hugging Face, you can skip preference pairs and training (sections **2–6**):

1. Set `LOAD_ORPO_FROM_HF = True` in the config cell (prevents accidentally running training cells).
2. Run install → login → config → **the next code cell** → section **7** (eval).

This reconstructs the stack the same way as training: **base → merge SFT → attach ORPO LoRA** from `ORPO_REPO`.

In [ ]:
if not LOAD_ORPO_FROM_HF:
    print("Skipping HF ORPO load (LOAD_ORPO_FROM_HF=False). Run sections 2–6 to train, or set the flag and re-run this cell.")
else:
    import os
    os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

    from transformers import AutoModelForCausalLM, AutoTokenizer, AutoModel
    from peft import PeftModel
    import torch

    MAX_SEQ_LENGTH = 1024

    try:
        _base = AutoModelForCausalLM.from_pretrained(
            BASE_MODEL_ID,
            dtype=torch.float16,
            device_map="auto",
            trust_remote_code=True,
        )
    except Exception as e:
        print(f"AutoModelForCausalLM failed ({e}), trying AutoModel...")
        _base = AutoModel.from_pretrained(
            BASE_MODEL_ID,
            dtype=torch.float16,
            device_map="auto",
            trust_remote_code=True,
        )

    if next(_base.parameters()).dtype not in (torch.float16, torch.float32):
        _base = _base.half()

    _sft = PeftModel.from_pretrained(_base, SFT_REPO)
    _merged = _sft.merge_and_unload()
    del _sft, _base

    model = PeftModel.from_pretrained(_merged, ORPO_REPO)
    print(f"Loaded ORPO LoRA on merged SFT: {ORPO_REPO} (SFT: {SFT_REPO})")

    tokenizer = AutoTokenizer.from_pretrained(ORPO_REPO, trust_remote_code=True)
    CHAT_TEMPLATE_NO_THINK = (
        "{%- for message in messages %}"
        "<|im_start|>{{ message.role }}\n"
        "{{ message.content }}<|im_end|>\n"
        "{%- endfor %}"
        "{%- if add_generation_prompt %}"
        "<|im_start|>assistant\n"
        "{%- endif %}"
    )
    tokenizer.chat_template = CHAT_TEMPLATE_NO_THINK
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    print(f"Model: {model.__class__.__name__}, dtype={next(model.parameters()).dtype}")

    # So section 7 (eval) can upload generations without running the push cell.
    from huggingface_hub import HfApi
    HF_REPO = ORPO_REPO
    api = HfApi()

## 2. Load Preference Pairs

In [ ]:
import json
import os
import glob as _glob

# Pick filename. PAIRS_VERSION is set in the config cell — defaults to v14 (1450 labeled pairs).
PAIRS_FILE = (
    f"preference_pairs_{PAIRS_VERSION}_multi.jsonl"
    if USE_MULTI_PAIRS
    else "preference_pairs.jsonl"
)

_candidates = [
    f"/kaggle/input/businessgpt-eval/{PAIRS_FILE}",
    f"/kaggle/input/businessgpt-eval/eval/{PAIRS_FILE}",
    f"eval/{PAIRS_FILE}",
    PAIRS_FILE,
]
_pairs_path = next((p for p in _candidates if os.path.isfile(p)), None)

if _pairs_path is None:
    _matches = _glob.glob(f"/kaggle/input/**/{PAIRS_FILE}", recursive=True)
    if _matches:
        _pairs_path = _matches[0]
        print(f"Found {PAIRS_FILE} via glob: {_pairs_path}")

if _pairs_path is None:
    raise FileNotFoundError(
        f"{PAIRS_FILE} not found. For multi pairs: run pairwise_ui_multi + "
        f"build_dpo_from_multi in the bench notebook, then version the businessgpt-eval "
        f"Kaggle dataset (`kaggle datasets version -p eval`)."
    )

with open(_pairs_path, encoding="utf-8") as f:
    pairs = [json.loads(line) for line in f if line.strip()]
print(f"Loaded {len(pairs)} preference pairs from {_pairs_path}")

# Stats
from collections import Counter
print(f"  by category: {Counter(p['category'] for p in pairs)}")
print(f"  by tier:     {Counter(p.get('tier', 'normal') for p in pairs)}")

# Apply super-tier weighting by replication.
weighted = []
for p in pairs:
    weighted.append(p)
    if p.get("tier") == "super" and SUPER_WEIGHT > 1:
        for _ in range(SUPER_WEIGHT - 1):
            weighted.append(p)
print(f"After super-weighting (×{SUPER_WEIGHT}): {len(weighted)} pairs")

## 3. Load Base Model + SFT Adapter (merge)

In [ ]:
import os
# Set before any torch/cuda init to reduce fragmentation (see PyTorch CUDA memory notes).
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

if LOAD_ORPO_FROM_HF:
    raise RuntimeError(
        "LOAD_ORPO_FROM_HF is True: skip this cell. Use 'Load ORPO from Hugging Face' above, then section 7 (eval)."
    )

from transformers import AutoModelForCausalLM, AutoTokenizer, AutoModel
from peft import PeftModel
import torch

# ORPO is lighter than DPO (no precompute_ref). 9B fp16 fits T4×2 comfortably.
# 2B uses 1536 for headroom; 9B uses 1024 — same as DPO's tuned numbers.
MAX_SEQ_LENGTH = 1024 if _MODEL_SIZE == "9b" else 1536

# ORPO has no precompute_ref step (the DPO fp16 NaN trigger on Qwen3.5 hybrid
# linear_attn). fp16 is safe here. Keep fp32 fallback for 2B for symmetry with
# the DPO-tested setup, but it shouldn't be needed.
ORPO_DTYPE = torch.float16

try:
    base_model = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL_ID,
        dtype=ORPO_DTYPE,
        device_map="auto",
        trust_remote_code=True,
    )
except Exception as e:
    print(f"AutoModelForCausalLM failed ({e}), trying AutoModel...")
    base_model = AutoModel.from_pretrained(
        BASE_MODEL_ID,
        dtype=ORPO_DTYPE,
        device_map="auto",
        trust_remote_code=True,
    )

if next(base_model.parameters()).dtype != ORPO_DTYPE:
    print(f"Casting from {next(base_model.parameters()).dtype} to {ORPO_DTYPE}...")
    base_model = base_model.half()

# Apply SFT adapter on top, then merge into the base. The merged model becomes
# the starting point for the new ORPO LoRA.
sft = PeftModel.from_pretrained(base_model, SFT_REPO)
model = sft.merge_and_unload()
print(f"Merged SFT adapter from {SFT_REPO} into base.")

tokenizer = AutoTokenizer.from_pretrained(SFT_REPO, trust_remote_code=True)
CHAT_TEMPLATE_NO_THINK = (
    "{%- for message in messages %}"
    "<|im_start|>{{ message.role }}\n"
    "{{ message.content }}<|im_end|>\n"
    "{%- endfor %}"
    "{%- if add_generation_prompt %}"
    "<|im_start|>assistant\n"
    "{%- endif %}"
)
tokenizer.chat_template = CHAT_TEMPLATE_NO_THINK
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f"Model: {model.__class__.__name__}, dtype={next(model.parameters()).dtype}")
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"MAX_SEQ_LENGTH: {MAX_SEQ_LENGTH}")

## 4. Apply DPO LoRA

In [ ]:
from peft import LoraConfig, get_peft_model

if LOAD_ORPO_FROM_HF:
    raise RuntimeError("LOAD_ORPO_FROM_HF is True: skip this cell (ORPO adapter already loaded from HF).")

# Same target_modules restriction as DPO: Qwen3.5 has hybrid linear_attn layers
# that go NaN under fp16 backward over long sequences. ORPO is theoretically safer
# (no precompute_ref step), but keep the exclusion as a belt-and-suspenders measure.
# Standard SFT trained "all-linear" — those weights stay baked in via merge_and_unload.
ORPO_TARGET_MODULES = [
    # full self-attention
    "q_proj", "k_proj", "v_proj", "o_proj",
    # MLP
    "gate_proj", "up_proj", "down_proj",
]

orpo_lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=ORPO_TARGET_MODULES,
    lora_dropout=0.05,
    use_dora=False,
    task_type="CAUSAL_LM",
)
model = get_peft_model(model, orpo_lora_config)
model.gradient_checkpointing_enable()

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"ORPO LoRA: {trainable:,} / {total:,} ({trainable/total*100:.2f}%) trainable")
print(f"Targets: {ORPO_TARGET_MODULES}  (linear_attn intentionally excluded)")

## 5. DPO Trainer

In [ ]:
from trl import ORPOTrainer, ORPOConfig
from datasets import Dataset

if LOAD_ORPO_FROM_HF:
    raise RuntimeError("LOAD_ORPO_FROM_HF is True: skip ORPO trainer (dataset + training).")

# --- Pre-flight: verify ORPOConfig signature (trl ships param names inconsistently across versions) ---
import inspect as _inspect
_orpo_sig = _inspect.signature(ORPOConfig.__init__)
_orpo_params = set(_orpo_sig.parameters)
print(f"ORPOConfig params (trl version): {len(_orpo_params)} total")
for _expected in ("beta", "max_length", "max_prompt_length"):
    if _expected not in _orpo_params:
        # try alternative names
        _alts = [p for p in _orpo_params if _expected in p]
        print(f"  ⚠️ '{_expected}' not found; alternatives: {_alts}")
    else:
        print(f"  ✓ '{_expected}' present")

import random as _random
_random.seed(42)
_random.shuffle(weighted)
val_size = max(20, int(len(weighted) * 0.05))
val_pairs = weighted[:val_size]
train_pairs = weighted[val_size:]

train_ds = Dataset.from_list(train_pairs)
val_ds   = Dataset.from_list(val_pairs)
print(f"Train: {len(train_ds)}  Val: {len(val_ds)}")

# --- Sanity-check pairs BEFORE expensive training ---
_n_identical, _n_short = 0, 0
for p in train_pairs:
    c = p["chosen"][-1]["content"].strip()
    r = p["rejected"][-1]["content"].strip()
    if c == r:
        _n_identical += 1
    if len(c) < 2 or len(r) < 2:
        _n_short += 1
print(f"Sanity: identical chosen/rejected = {_n_identical}, very-short responses = {_n_short}")
assert _n_identical < len(train_pairs) * 0.05, \
    "Too many identical chosen/rejected pairs — ORPO has no signal."

# Length-bias diagnostic (ORPO doesn't suffer from it the way DPO does — the SFT-on-chosen
# term anchors absolute log-probs — but log it anyway for awareness).
_chosen_lens = [len(p["chosen"][-1]["content"]) for p in train_pairs]
_rejected_lens = [len(p["rejected"][-1]["content"]) for p in train_pairs]
_ratio = sum(_chosen_lens) / max(sum(_rejected_lens), 1)
print(f"Length: chosen mean={sum(_chosen_lens)/len(_chosen_lens):.1f}, "
      f"rejected mean={sum(_rejected_lens)/len(_rejected_lens):.1f}, "
      f"ratio={_ratio:.2f}  (ORPO is more length-robust than DPO)")

NUM_EPOCHS = 2
BATCH_SIZE = 1
GRAD_ACCUM = 8
EFFECTIVE_BATCH = BATCH_SIZE * GRAD_ACCUM
STEPS_PER_EPOCH = max(1, len(train_ds) // EFFECTIVE_BATCH)
EVAL_STEPS = max(1, STEPS_PER_EPOCH // 4)

# ORPO hyperparameters (vs DPO):
# - learning_rate 1e-5: 20× higher than DPO's 5e-7. ORPO has no ref-model drift,
#   so the policy can move further per step without runaway. 1e-5 is the value
#   from the ORPO paper.
# - beta 0.1: in trl ORPOConfig, "beta" is the odds-ratio preference weight
#   (called λ in the paper). 0.1 is paper default — keeps SFT loss dominant.
# - no precompute_ref_log_probs (ORPO has no ref model at all).
# - max_prompt_length=768 leaves 256 tokens of response under max_length=1024.
# - fp16 safe (no precompute_ref step = no Qwen3.5 hybrid NaN trigger).
orpo_args = ORPOConfig(
    output_dir="orpo_outputs",
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=GRAD_ACCUM,
    num_train_epochs=NUM_EPOCHS,
    learning_rate=1e-5,
    beta=0.1,
    warmup_ratio=0.1,
    max_grad_norm=1.0,
    lr_scheduler_type="cosine",
    max_length=MAX_SEQ_LENGTH,
    max_prompt_length=int(MAX_SEQ_LENGTH * 0.75),
    truncation_mode="keep_start",
    logging_steps=5,
    eval_strategy="steps",
    eval_steps=EVAL_STEPS,
    save_strategy="steps",
    save_steps=EVAL_STEPS,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    fp16=True,
    bf16=False,
    gradient_checkpointing=True,
    seed=3407,
    report_to="none",
)


# --- Stability callback: catch NaN/Inf loss + reward-margin explosion early ---
# Same callback as DPO. ORPO logs rewards/margins similarly.
from transformers import TrainerCallback
import math

class _StabilityAbortCallback(TrainerCallback):
    def on_log(self, args, state, control, logs=None, **kwargs):
        if not logs:
            return
        loss = logs.get("loss")
        if loss is not None and (math.isnan(loss) or math.isinf(loss)):
            print(f"\n⚠️ NaN/Inf training loss at step {state.global_step} — aborting.")
            control.should_training_stop = True
            return
        margin = logs.get("rewards/margins")
        if margin is not None and abs(margin) > 50:
            print(f"\n⚠️ rewards/margins = {margin:.2f} (>50) at step {state.global_step} — runaway, aborting.")
            control.should_training_stop = True


trainer = ORPOTrainer(
    model=model,
    args=orpo_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    processing_class=tokenizer,
    callbacks=[_StabilityAbortCallback()],
)

print(f"Training config:")
print(f"  Train pairs: {len(train_ds)} (effective batch {EFFECTIVE_BATCH})")
print(f"  Steps/epoch: {STEPS_PER_EPOCH}, total: {STEPS_PER_EPOCH * NUM_EPOCHS}")
print(f"  beta={orpo_args.beta}  lr={orpo_args.learning_rate}  super_weight={SUPER_WEIGHT}")
print(f"  precision: fp16 (no precompute_ref → fp16 NaN risk avoided)")

In [ ]:
if LOAD_ORPO_FROM_HF:
    raise RuntimeError("LOAD_ORPO_FROM_HF is True: skip training.")
trainer_stats = trainer.train()
print(f"\nORPO training complete!")
print(f"  Total steps: {trainer_stats.global_step}")
print(f"  Final loss:  {trainer_stats.training_loss:.4f}")

# --- Post-training NaN check ---
# If weights NaN'd, generation will collapse to "!!!!!!!". Catch it here, not at eval.
import torch
nan_params = [n for n, p in model.named_parameters() if torch.isnan(p).any()]
inf_params = [n for n, p in model.named_parameters() if torch.isinf(p).any()]
if nan_params or inf_params:
    print(f"\n⚠️ MODEL CORRUPTED:")
    print(f"  {len(nan_params)} params contain NaN: {nan_params[:3]}{'...' if len(nan_params)>3 else ''}")
    print(f"  {len(inf_params)} params contain Inf: {inf_params[:3]}{'...' if len(inf_params)>3 else ''}")
    print("  → Lower learning_rate (5e-6 or 1e-6), shrink max_length, or fall back to fp32.")
    raise RuntimeError("ORPO produced NaN/Inf weights — do NOT push, retrain with safer hyperparams.")
else:
    print(f"  Sanity: no NaN/Inf in trainable params.")

# Quick generation smoke test — must produce something, not "!!!!".
import random as _r
_r.seed(0)
_test_prompt = "Некит Русанов: пиздец как дела?"
_inp = tokenizer(
    f"<|im_start|>system\n<|im_end|>\n<|im_start|>user\n{_test_prompt}<|im_end|>\n<|im_start|>assistant\n",
    return_tensors="pt", add_special_tokens=False,
).to(next(model.parameters()).device)
with torch.no_grad():
    _out = model.generate(**_inp, max_new_tokens=30, do_sample=False,
                          pad_token_id=tokenizer.eos_token_id)
_smoke = tokenizer.decode(_out[0][_inp["input_ids"].shape[1]:], skip_special_tokens=True)
print(f"\nSmoke generation:\n  prompt: {_test_prompt}\n  response: {_smoke!r}")
if _smoke.strip() in ("", "!", "!!", "!!!") or _smoke.strip().count("!") > 5:
    raise RuntimeError("Smoke test failed — model collapsed to dead token. Do not push.")

## 6. Save & Push

In [ ]:
model.save_pretrained(SAVE_DIR, safe_serialization=True)
tokenizer.save_pretrained(SAVE_DIR)
print(f"ORPO adapter saved to {SAVE_DIR}/")

import os
for f in sorted(os.listdir(SAVE_DIR)):
    size = os.path.getsize(f"{SAVE_DIR}/{f}")
    if size > 1024 * 1024:
        print(f"  {f}: {size/1024/1024:.1f} MB")
    else:
        print(f"  {f}: {size/1024:.1f} KB")

In [ ]:
from huggingface_hub import HfApi

HF_REPO = ORPO_REPO
api = HfApi()
api.create_repo(HF_REPO, exist_ok=True)
api.upload_folder(
    folder_path=SAVE_DIR,
    repo_id=HF_REPO,
    commit_message=f"Upload BusinessGPT {ORPO_VERSION} ORPO adapter (base={BASE_VERSION}, beta=0.1, lr=1e-5, pairs={PAIRS_VERSION}_multi)",
)
print(f"Pushed to https://huggingface.co/{HF_REPO}")

## 7. Eval: generate on golden_prompts set

Same eval cell as `training.ipynb` — produces `eval/generations_<DPO_VERSION>.json`
that the bench notebook can compare against the SFT base via
`pairwise_ui("v13", "v13-dpo")` and `guardrails_table(...)`.

In [ ]:
import io
import contextlib
import re as _re

EVAL_DIR = "/kaggle/working/eval"
os.makedirs(EVAL_DIR, exist_ok=True)
VERSION = ORPO_VERSION

# Locate golden_prompts.json
_candidates = [
    "/kaggle/input/businessgpt-eval/golden_prompts.json",
    "/kaggle/input/businessgpt-eval/eval/golden_prompts.json",
    "eval/golden_prompts.json",
    "golden_prompts.json",
]
_golden_path = next((p for p in _candidates if os.path.isfile(p)), None)
if _golden_path is None:
    _matches = _glob.glob("/kaggle/input/**/golden_prompts.json", recursive=True)
    if _matches:
        _golden_path = _matches[0]
if _golden_path is None:
    from huggingface_hub import hf_hub_download
    _golden_path = hf_hub_download(repo_id=HF_REPO, filename="eval/golden_prompts.json")

with open(_golden_path, encoding="utf-8") as f:
    golden = json.load(f)
print(f"Loaded {len(golden)} prompts from {_golden_path}")

SYSTEM_PROMPT_INFER = (
    "Ты BusinessGPT. Пиши как студент в мессенджере: коротко, дерзко, ахуевше, "
    "по-пидорски. Часто вставляй слова-паразиты: бля, нах, блять, ёпт, пиздец."
)

def _format_chat(messages):
    return "\n".join(f"<|im_start|>{m['role']}\n{m['content']}<|im_end|>" for m in messages)

_im_start_id = tokenizer.convert_tokens_to_ids("<|im_start|>")
_im_end_id   = tokenizer.convert_tokens_to_ids("<|im_end|>")
_vocab_sz = getattr(tokenizer, "vocab_size", None) or len(tokenizer)

def _valid_tid(tid):
    return tid is not None and isinstance(tid, int) and tid != tokenizer.unk_token_id and 0 <= tid < _vocab_sz

_eos_ids = [x for x in (_im_end_id, _im_start_id) if _valid_tid(x)]
if not _eos_ids:
    _eos_ids = [int(tokenizer.eos_token_id)]
_eos_for_generate = _eos_ids[0] if len(_eos_ids) == 1 else _eos_ids
_pad_id = tokenizer.pad_token_id or tokenizer.eos_token_id

_input_device = next(model.parameters()).device
model.eval()

# Same 4-candidate setup as eval_only.ipynb / training.ipynb / pairwise_ui_multi.
# Required so the bench can compare v16-orpo vs v16 with pairwise_ui_multi.
CANDIDATE_PARAMS = [
    {"idx": 0, "temperature": 0.7,  "top_p": 0.9,  "top_k": 50, "repetition_penalty": 1.1,  "seed": 42},
    {"idx": 1, "temperature": 0.95, "top_p": 0.9,  "top_k": 50, "repetition_penalty": 1.1,  "seed": 43},
    {"idx": 2, "temperature": 1.1,  "top_p": 0.92, "top_k": 60, "repetition_penalty": 1.1,  "seed": 44},
    {"idx": 3, "temperature": 1.3,  "top_p": 0.95, "top_k": 80, "repetition_penalty": 1.05, "seed": 45},
]
DEFAULT_CANDIDATE_IDX = 1
MAX_NEW_TOKENS = 200

def _chat_with_params(context, *, max_new_tokens, temperature, top_p, top_k,
                      repetition_penalty, seed):
    if context and isinstance(context[0], str):
        messages = [
            {"role": "system", "content": SYSTEM_PROMPT_INFER},
            {"role": "user", "content": "\n".join(context)},
        ]
    else:
        messages = [{"role": "system", "content": SYSTEM_PROMPT_INFER}] + list(context)
    input_text = _format_chat(messages) + "\n<|im_start|>assistant\n"
    inputs = tokenizer(input_text, return_tensors="pt", add_special_tokens=False).to(_input_device)
    torch.manual_seed(seed)
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=temperature, top_p=top_p, top_k=top_k,
            repetition_penalty=repetition_penalty,
            eos_token_id=_eos_for_generate,
            pad_token_id=_pad_id,
        )
    g = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=False)
    g = _re.sub(r"<think>.*?</think>\s*", "", g, flags=_re.DOTALL).strip()
    g = _re.sub(r"<\|im_start\|>.*", "", g, flags=_re.DOTALL)
    g = _re.sub(r"<\|im_end\|>", "", g)
    g = _re.sub(r"^<bot>\s*", "", g).strip()
    return g

# --- Generate 4 candidates per prompt ---
multi = []
for i, p in enumerate(golden):
    candidates = []
    for params in CANDIDATE_PARAMS:
        with contextlib.redirect_stdout(io.StringIO()):
            response = _chat_with_params(
                p["context"], max_new_tokens=MAX_NEW_TOKENS,
                temperature=params["temperature"], top_p=params["top_p"],
                top_k=params["top_k"], repetition_penalty=params["repetition_penalty"],
                seed=params["seed"],
            )
        candidates.append({**params, "response": response})
    multi.append({
        "id": p["id"], "category": p["category"],
        "held_out": p.get("held_out", False), "context": p["context"],
        "version": VERSION, "candidates": candidates,
    })
    if (i + 1) % 25 == 0 or (i + 1) == len(golden):
        print(f"  {i+1}/{len(golden)}")

multi_path = os.path.join(EVAL_DIR, f"generations_{VERSION}_multi.json")
with open(multi_path, "w", encoding="utf-8") as f:
    json.dump(multi, f, ensure_ascii=False, indent=1)

# Legacy single-candidate file: pick idx=1 (temp 0.95) for backward-compat with bench guardrails/pairwise.
single = []
for entry in multi:
    default = next(c for c in entry["candidates"] if c["idx"] == DEFAULT_CANDIDATE_IDX)
    single.append({
        "id": entry["id"], "category": entry["category"],
        "held_out": entry["held_out"], "context": entry["context"],
        "response": default["response"], "version": VERSION,
        "sampling_params": {k: default[k] for k in ("temperature", "top_p", "top_k", "repetition_penalty")},
        "seed": default["seed"],
    })

single_path = os.path.join(EVAL_DIR, f"generations_{VERSION}.json")
with open(single_path, "w", encoding="utf-8") as f:
    json.dump(single, f, ensure_ascii=False, indent=1)

print(f"\nSaved {len(multi)} multi-candidate generations to {multi_path}")
print(f"Saved {len(single)} default-candidate generations to {single_path}")

for path, label in [(single_path, "single"), (multi_path, "multi")]:
    try:
        api.upload_file(
            path_or_fileobj=path,
            path_in_repo=f"eval/{os.path.basename(path)}",
            repo_id=HF_REPO,
            commit_message=f"Upload eval generations ({VERSION}, {label})",
        )
        print(f"  HF upload OK: {os.path.basename(path)}")
    except Exception as e:
        print(f"  HF upload failed for {label}: {e}")